In [0]:
dbutils.library.restartPython()

In [0]:
%run ../setup_path

In [0]:
from pyspark.sql import functions as F
import yaml
from utils.data_transformation import apply_data_transformation

In [0]:
yaml_file = yaml.load(open("./mapping/silver_ocm_poi.yaml"), Loader=yaml.FullLoader)

PROJECT_ID = yaml_file['project_id']
SOURCE_TABLE = yaml_file['source_table']
TARGET_TABLE = yaml_file['target_table']
TARGET_SCHEMA = [c['target'] for c in yaml_file['schema']]


In [0]:
RUN_ID = 1793953962649

ocm_poi_df = (
                spark.table(f"`{PROJECT_ID}`.{SOURCE_TABLE}")
                .filter(F.col("_run_id") == RUN_ID)
)


In [0]:
ocm_poi_df = apply_data_transformation(ocm_poi_df, yaml_file)

In [0]:
RUN_ID

In [0]:
spark.sql(f"""DELETE FROM `{PROJECT_ID}`.{TARGET_TABLE} WHERE `_run_id` = {RUN_ID}""")

In [0]:
ocm_poi_df = ocm_poi_df.select(TARGET_SCHEMA)
ocm_poi_df.write.mode("append").saveAsTable(f"`{PROJECT_ID}`.{TARGET_TABLE}")